# Hemicellulose Yield Optimization
## Bayesian Optimization with Gaussian Process Surrogate Model

**Objective:** Maximize hemicellulose extraction yield (R1 %)

**Factors:**
- **A: Temperature (°C)** — range [170, 220]
- **B: Time (min)** — range [10, 30]
- **C: S/L Ratio** — range [10, 20]

**RSM Quadratic Model:**
$$Y = 30.0 + 4.1A + 3.0B + 2.7C - 1.7AB - 1.5AC - 1.3BC - 4.8A^2 - 4.0B^2 - 3.5C^2$$
(coded variables; A=(T−195)/25, B=(t−20)/10, C=(SL−15)/5)


In [ ]:
# ── 0. Imports ───────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from mpl_toolkits.mplot3d import Axes3D
import seaborn as sns
from scipy.stats import norm
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, ConstantKernel, WhiteKernel
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score, KFold
from sklearn.metrics import r2_score, mean_squared_error
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi': 120,
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
})
print('All imports successful ✓')

In [ ]:
# ── 1. RSM Quadratic Model & Data Generation ────────────────────────────────

def coded(T, t, SL):
    """Convert actual to coded variables."""
    A = (T - 195) / 25
    B = (t - 20)  / 10
    C = (SL - 15) /  5
    return A, B, C

def rsm_model(T, t, SL):
    """RSM quadratic model from Design-Expert output."""
    A, B, C = coded(T, t, SL)
    Y = (30.0
         + 4.1*A  + 3.0*B  + 2.7*C
         - 1.7*A*B - 1.5*A*C - 1.3*B*C
         - 4.8*A**2 - 4.0*B**2 - 3.5*C**2)
    return float(np.clip(Y, 0, 100))

# 20 real experimental points
real_pts = [
    (195, 10, 15), (170, 30, 10), (195, 30, 15), (220, 10, 20), (220, 30, 20),
    (220, 20, 15), (195, 20, 15), (195, 20, 10), (195, 20, 15), (170, 30, 20),
    (195, 20, 15), (195, 20, 15), (220, 10, 10), (170, 20, 15), (220, 30, 10),
    (195, 20, 15), (170, 10, 20), (170, 10, 10), (195, 20, 20), (195, 20, 15),
]
real_yields = [None, 20.5, 26.0, 27.1, 28.6, 26.9, 30.0, 29.5, 30.0, 23.5,
               30.0, 30.0, 23.2, 21.8, 26.3, 30.0, 20.7, 18.5, 26.0, 30.0]

df_real = pd.DataFrame(real_pts, columns=['Temperature_C','Time_Min','SL_Ratio'])
df_real['Yield_Experimental'] = real_yields
df_real['Yield_Experimental'] = df_real['Yield_Experimental'].fillna(
    df_real.apply(lambda r: rsm_model(r.Temperature_C, r.Time_Min, r.SL_Ratio), axis=1))
df_real['Type'] = 'Experimental'

# Load or regenerate 200-point dataset
try:
    df = pd.read_csv('hemicellulose_200pts.csv', index_col='ID')
    print('Loaded 200-pt CSV ✓')
except FileNotFoundError:
    np.random.seed(42)
    rows = []
    for T in np.linspace(170, 220, 10):
        for t in np.linspace(10, 30, 9):
            for SL in np.linspace(10, 20, 10):
                Y = rsm_model(T, t, SL) + np.random.normal(0, 0.3)
                rows.append([round(T,1), round(t,1), round(SL,1), round(np.clip(Y,0,100),2), 'Synthetic'])
    df_synth = pd.DataFrame(rows, columns=['Temperature_C','Time_Min','SL_Ratio','Yield_pct','Type'])
    df_real_out = df_real[['Temperature_C','Time_Min','SL_Ratio']].copy()
    df_real_out['Yield_pct'] = df_real['Yield_Experimental'].values
    df_real_out['Type'] = 'Experimental'
    df = pd.concat([df_real_out, df_synth.sample(180, random_state=42)], ignore_index=True)
    df.index += 1; df.index.name='ID'
    print('Generated 200-pt dataset ✓')

print(f'Dataset shape: {df.shape}')
df.head()

In [ ]:
# ── 2. Dataset Summary ───────────────────────────────────────────────────────
print('=== Dataset Statistics ===')
print(df.describe().round(3))
print(f"\nExperimental points : {(df['Type']=='Experimental').sum()}")
print(f"Synthetic points    : {(df['Type']=='Synthetic').sum()}")

In [ ]:
# ── 3. Prepare Training Data ─────────────────────────────────────────────────
X = df[['Temperature_C','Time_Min','SL_Ratio']].values
y = df['Yield_pct'].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Gaussian Process with Matérn 5/2 kernel (standard for Bayesian optimization)
kernel = ConstantKernel(1.0) * Matern(nu=2.5) + WhiteKernel(noise_level=0.1)
gp = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=10,
                               normalize_y=True, random_state=42)
gp.fit(X_scaled, y)

y_pred = gp.predict(X_scaled)
r2     = r2_score(y, y_pred)
rmse   = np.sqrt(mean_squared_error(y, y_pred))

# Cross-validation
cv = KFold(n_splits=5, shuffle=True, random_state=42)
cv_r2 = cross_val_score(gp, X_scaled, y, cv=cv, scoring='r2').mean()

print('=== Gaussian Process Model Performance ===')
print(f'R²          : {r2:.4f}')
print(f'RMSE        : {rmse:.4f} %')
print(f'CV R² (5-fold): {cv_r2:.4f}')

In [ ]:
# ── 4. Bayesian Optimization Engine ─────────────────────────────────────────

def expected_improvement(X_candidates, gp, y_best, xi=0.01):
    """Expected Improvement acquisition function."""
    mu, sigma = gp.predict(X_candidates, return_std=True)
    sigma = np.maximum(sigma, 1e-9)
    z = (mu - y_best - xi) / sigma
    ei = (mu - y_best - xi) * norm.cdf(z) + sigma * norm.pdf(z)
    return ei

def upper_confidence_bound(X_candidates, gp, kappa=2.576):
    """Upper Confidence Bound acquisition function."""
    mu, sigma = gp.predict(X_candidates, return_std=True)
    return mu + kappa * sigma

def bayesian_optimization(gp, scaler, n_iterations=50, acquisition='EI',
                          T_bounds=(170,220), t_bounds=(10,30), SL_bounds=(10,20),
                          n_candidates=5000, xi=0.01, kappa=2.576, random_state=42):
    """
    Pure Bayesian Optimization using GP surrogate + acquisition function.
    Returns history of (T, t, SL, predicted_yield) at each iteration.
    """
    rng = np.random.RandomState(random_state)
    y_best = gp.predict(scaler.transform([[195, 20, 15]]))[0]  # center point start

    history = []
    best_so_far = []

    for i in range(n_iterations):
        # Random candidates in the design space
        candidates = np.column_stack([
            rng.uniform(*T_bounds,  n_candidates),
            rng.uniform(*t_bounds,  n_candidates),
            rng.uniform(*SL_bounds, n_candidates),
        ])
        cand_scaled = scaler.transform(candidates)

        if acquisition == 'EI':
            scores = expected_improvement(cand_scaled, gp, y_best, xi=xi)
        else:
            scores = upper_confidence_bound(cand_scaled, gp, kappa=kappa)

        best_idx = np.argmax(scores)
        best_pt  = candidates[best_idx]
        T_opt, t_opt, SL_opt = best_pt

        # Query true model (RSM) as oracle
        y_new = rsm_model(T_opt, t_opt, SL_opt)
        y_best = max(y_best, y_new)

        history.append({'Iteration': i+1,
                        'Temperature_C': round(T_opt, 2),
                        'Time_Min':      round(t_opt, 2),
                        'SL_Ratio':      round(SL_opt, 2),
                        'Predicted_Yield': round(y_new, 4),
                        'Best_So_Far':   round(y_best, 4)})
        best_so_far.append(y_best)

    return pd.DataFrame(history), best_so_far

print('Running Bayesian Optimization (EI acquisition)...')
df_bo_ei, bsf_ei = bayesian_optimization(gp, scaler, n_iterations=60, acquisition='EI')

print('Running Bayesian Optimization (UCB acquisition)...')
df_bo_ucb, bsf_ucb = bayesian_optimization(gp, scaler, n_iterations=60, acquisition='UCB')

# Best result
best_row = df_bo_ei.loc[df_bo_ei['Predicted_Yield'].idxmax()]
print('\n=== OPTIMAL CONDITIONS (EI) ===')
print(f"Temperature : {best_row.Temperature_C:.2f} °C")
print(f"Time        : {best_row.Time_Min:.2f} min")
print(f"S/L Ratio   : {best_row.SL_Ratio:.2f}")
print(f"Max Yield   : {best_row.Predicted_Yield:.4f} %")

In [ ]:
# ── 5. PLOT 1: Predicted vs Actual ──────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# GP model on full 200-pt dataset
ax = axes[0]
ax.scatter(y, y_pred, c=y, cmap='RdYlGn', s=60, alpha=0.7, edgecolors='grey', linewidth=0.3)
lims = [min(y.min(), y_pred.min())-0.5, max(y.max(), y_pred.max())+0.5]
ax.plot(lims, lims, 'k--', lw=1.5, label='Perfect fit')
ax.set_xlabel('Actual Yield (%)')
ax.set_ylabel('Predicted Yield (%)')
ax.set_title(f'Predicted vs Actual\nGP Model — R² = {r2:.4f}, RMSE = {rmse:.3f}%')
ax.set_xlim(lims); ax.set_ylim(lims)
ax.legend()
ax.grid(True, alpha=0.3)

# Residuals
ax = axes[1]
residuals = y - y_pred
ax.scatter(y_pred, residuals, c=y_pred, cmap='RdYlGn', s=60, alpha=0.7,
           edgecolors='grey', linewidth=0.3)
ax.axhline(0, color='k', linestyle='--', lw=1.5)
ax.set_xlabel('Predicted Yield (%)')
ax.set_ylabel('Residual (%)')
ax.set_title('Residuals Plot')
ax.grid(True, alpha=0.3)

plt.suptitle('Model Validation', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('predicted_vs_actual.png', bbox_inches='tight', dpi=150)
plt.show()
print('Plot saved: predicted_vs_actual.png')

In [ ]:
# ── 6. PLOT 2: 3D Surface Plots ──────────────────────────────────────────────
fig = plt.figure(figsize=(18, 14))
cmap = 'RdYlGn'

combos = [
    ('Temperature_C', 'Time_Min',  'SL_Ratio',  15,   'S/L=15 (center)',   (170,220), (10,30)),
    ('Temperature_C', 'SL_Ratio',  'Time_Min',  20,   'Time=20 min (center)', (170,220), (10,20)),
    ('Time_Min',      'SL_Ratio',  'Temperature_C', 195, 'Temp=195°C (center)', (10,30), (10,20)),
]

for idx, (xname, yname, fixed_name, fixed_val, subtitle, xr, yr) in enumerate(combos):
    ax = fig.add_subplot(2, 3, idx+1, projection='3d')

    xi_arr = np.linspace(*xr, 40)
    yi_arr = np.linspace(*yr, 40)
    Xi, Yi = np.meshgrid(xi_arr, yi_arr)
    Zi = np.zeros_like(Xi)

    for i in range(Xi.shape[0]):
        for j in range(Xi.shape[1]):
            args = {xname: Xi[i,j], yname: Yi[i,j], fixed_name: fixed_val}
            Zi[i,j] = rsm_model(args['Temperature_C'], args['Time_Min'], args['SL_Ratio'])

    surf = ax.plot_surface(Xi, Yi, Zi, cmap=cmap, alpha=0.85, linewidth=0)
    ax.set_xlabel(xname.replace('_',' '))
    ax.set_ylabel(yname.replace('_',' '))
    ax.set_zlabel('Yield (%)')
    ax.set_title(f'3D Surface\n({subtitle})', fontsize=10)
    fig.colorbar(surf, ax=ax, shrink=0.4, pad=0.1)

    # Contour plot below
    ax2 = fig.add_subplot(2, 3, idx+4)
    cp = ax2.contourf(Xi, Yi, Zi, levels=20, cmap=cmap)
    ax2.contour(Xi, Yi, Zi, levels=10, colors='k', linewidths=0.5, alpha=0.5)
    fig.colorbar(cp, ax=ax2, label='Yield (%)')
    ax2.set_xlabel(xname.replace('_',' '))
    ax2.set_ylabel(yname.replace('_',' '))
    ax2.set_title(f'Contour Plot\n({subtitle})', fontsize=10)
    ax2.grid(True, alpha=0.2)

plt.suptitle('3D Surface & Contour Plots — Hemicellulose Yield (%)',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('surface_contour_plots.png', bbox_inches='tight', dpi=150)
plt.show()
print('Plot saved: surface_contour_plots.png')

In [ ]:
# ── 7. PLOT 3: Cube Plot ─────────────────────────────────────────────────────
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

# 8 corners of the cube
corners = [
    (170, 10, 10), (220, 10, 10), (170, 30, 10), (220, 30, 10),
    (170, 10, 20), (220, 10, 20), (170, 30, 20), (220, 30, 20),
]
corner_yields = [rsm_model(T, t, SL) for T,t,SL in corners]

# Center and axial points
center_pts = [(195, 20, 15)] * 5
axial_pts  = [(170,20,15),(220,20,15),(195,10,15),(195,30,15),(195,20,10),(195,20,20)]
axial_yields = [rsm_model(T,t,SL) for T,t,SL in axial_pts]

# Draw cube edges
edges = [
    (0,1),(2,3),(4,5),(6,7),  # parallel to Temp axis
    (0,2),(1,3),(4,6),(5,7),  # parallel to Time axis
    (0,4),(1,5),(2,6),(3,7),  # parallel to SL axis
]
corners_arr = np.array(corners)
for e in edges:
    ax.plot(*zip(corners_arr[e[0]], corners_arr[e[1]]),
            color='steelblue', lw=1.2, alpha=0.6)

# Scatter corners with yield color
T_c, t_c, SL_c = zip(*corners)
sc = ax.scatter(T_c, t_c, SL_c, c=corner_yields, cmap='RdYlGn',
                s=200, zorder=5, edgecolors='black', linewidth=0.8, vmin=15, vmax=32)
for (T,t,SL), Y in zip(corners, corner_yields):
    ax.text(T, t, SL+0.4, f'{Y:.2f}%', fontsize=7.5, ha='center', va='bottom')

# Axial / star points
T_a, t_a, SL_a = zip(*axial_pts)
ax.scatter(T_a, t_a, SL_a, c=axial_yields, cmap='RdYlGn', s=150, marker='^',
           vmin=15, vmax=32, edgecolors='black', linewidth=0.8, zorder=6)

# Center point
cY = rsm_model(195, 20, 15)
ax.scatter([195], [20], [15], c=[cY], cmap='RdYlGn', s=300, marker='*',
           vmin=15, vmax=32, edgecolors='black', linewidth=0.8, zorder=7)
ax.text(195, 20, 15.6, f'{cY:.2f}%', fontsize=8, ha='center', color='darkred', fontweight='bold')

fig.colorbar(sc, ax=ax, label='Predicted Yield (%)', shrink=0.6)
ax.set_xlabel('Temperature (°C)')
ax.set_ylabel('Time (min)')
ax.set_zlabel('S/L Ratio')
ax.set_title('Cube Plot — Hemicellulose Yield (%)\nPredicted Values at Design Points',
             fontsize=12, fontweight='bold')
ax.view_init(elev=20, azim=35)
plt.tight_layout()
plt.savefig('cube_plot.png', bbox_inches='tight', dpi=150)
plt.show()
print('Plot saved: cube_plot.png')

In [ ]:
# ── 8. PLOT 4: Bayesian Optimization Convergence ─────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(df_bo_ei['Iteration'], df_bo_ei['Best_So_Far'],
        'b-o', markersize=4, lw=2, label='EI — Best so far')
ax.plot(df_bo_ucb['Iteration'], df_bo_ucb['Best_So_Far'],
        'r--s', markersize=4, lw=2, label='UCB — Best so far')
ax.axhline(df_bo_ei['Best_So_Far'].max(), color='blue', linestyle=':', alpha=0.6,
           label=f"EI max = {df_bo_ei['Best_So_Far'].max():.3f}%")
ax.set_xlabel('Iteration'); ax.set_ylabel('Best Yield Found (%)')
ax.set_title('Bayesian Optimization Convergence')
ax.legend(); ax.grid(True, alpha=0.3)

# Exploration path scatter
ax = axes[1]
sc = ax.scatter(df_bo_ei['Temperature_C'], df_bo_ei['Time_Min'],
                c=df_bo_ei['Predicted_Yield'], cmap='RdYlGn',
                s=60, alpha=0.8, edgecolors='grey', linewidth=0.3)
# Mark best
best_row = df_bo_ei.loc[df_bo_ei['Predicted_Yield'].idxmax()]
ax.scatter(best_row.Temperature_C, best_row.Time_Min,
           s=250, marker='*', c='gold', edgecolors='black', zorder=10,
           label=f"Optimum: T={best_row.Temperature_C:.1f}°C, t={best_row.Time_Min:.1f}min")
fig.colorbar(sc, ax=ax, label='Yield (%)')
ax.set_xlabel('Temperature (°C)'); ax.set_ylabel('Time (min)')
ax.set_title('BO Exploration Path (EI)\nColour = Predicted Yield')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

plt.suptitle('Bayesian Optimization Results', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('bayesian_optimization_convergence.png', bbox_inches='tight', dpi=150)
plt.show()
print('Plot saved: bayesian_optimization_convergence.png')

In [ ]:
# ── 9. PLOT 5: Acquisition Function Landscape ────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
y_best_current = df_bo_ei['Best_So_Far'].max()

pair_configs = [
    ('Temperature_C', 'Time_Min',   {'SL_Ratio': 15},   (170,220), (10,30)),
    ('Temperature_C', 'SL_Ratio',   {'Time_Min': 20},   (170,220), (10,20)),
    ('Time_Min',      'SL_Ratio',   {'Temperature_C':195},(10,30), (10,20)),
]

for ax, (xn, yn, fixed, xr, yr) in zip(axes, pair_configs):
    xi_g = np.linspace(*xr, 50)
    yi_g = np.linspace(*yr, 50)
    Xg, Yg = np.meshgrid(xi_g, yi_g)
    cands = []
    for xi_v, yi_v in zip(Xg.ravel(), Yg.ravel()):
        row = fixed.copy()
        row[xn] = xi_v; row[yn] = yi_v
        cands.append([row['Temperature_C'], row['Time_Min'], row['SL_Ratio']])
    cands_scaled = scaler.transform(cands)
    ei_vals = expected_improvement(cands_scaled, gp, y_best_current).reshape(Xg.shape)

    cp = ax.contourf(Xg, Yg, ei_vals, levels=25, cmap='plasma')
    ax.contour(Xg, Yg, ei_vals, levels=10, colors='white', linewidths=0.4, alpha=0.5)
    fig.colorbar(cp, ax=ax, label='EI Score')
    ax.set_xlabel(xn.replace('_',' ')); ax.set_ylabel(yn.replace('_',' '))
    fixed_str = ', '.join(f"{k.replace('_',' ')}={v}" for k,v in fixed.items())
    ax.set_title(f'Expected Improvement\n({fixed_str})')
    ax.grid(True, alpha=0.2)

plt.suptitle('Acquisition Function Landscape (EI)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('acquisition_landscape.png', bbox_inches='tight', dpi=150)
plt.show()
print('Plot saved: acquisition_landscape.png')

In [ ]:
# ── 10. PLOT 6: GP Uncertainty / Confidence Interval ────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (xn, yn, fixed, xr, yr) in zip(axes, pair_configs):
    xi_g = np.linspace(*xr, 50)
    yi_g = np.linspace(*yr, 50)
    Xg, Yg = np.meshgrid(xi_g, yi_g)
    cands = []
    for xi_v, yi_v in zip(Xg.ravel(), Yg.ravel()):
        row = fixed.copy(); row[xn] = xi_v; row[yn] = yi_v
        cands.append([row['Temperature_C'], row['Time_Min'], row['SL_Ratio']])
    cands_scaled = scaler.transform(cands)
    mu, sigma = gp.predict(cands_scaled, return_std=True)
    Mu = mu.reshape(Xg.shape)
    Sigma = sigma.reshape(Xg.shape)

    cp = ax.contourf(Xg, Yg, Sigma, levels=20, cmap='YlOrRd')
    ax.contour(Xg, Yg, Mu, levels=12, colors='black', linewidths=0.6,
               linestyles='solid', alpha=0.6)
    fig.colorbar(cp, ax=ax, label='GP Std Dev (uncertainty)')
    ax.set_xlabel(xn.replace('_',' ')); ax.set_ylabel(yn.replace('_',' '))
    fixed_str = ', '.join(f"{k.replace('_',' ')}={v}" for k,v in fixed.items())
    ax.set_title(f'GP Uncertainty\n({fixed_str})')
    ax.grid(True, alpha=0.2)

plt.suptitle('Gaussian Process Uncertainty Landscape\n(higher = less explored)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('gp_uncertainty.png', bbox_inches='tight', dpi=150)
plt.show()
print('Plot saved: gp_uncertainty.png')

In [ ]:
# ── 11. PLOT 7: Grid Search Heatmap Comparison ──────────────────────────────
T_vals  = np.linspace(170, 220, 50)
SL_vals = np.linspace(10, 20, 50)
t_fixed = [10, 20, 30]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, t_val in zip(axes, t_fixed):
    Z = np.array([[rsm_model(T, t_val, SL) for T in T_vals] for SL in SL_vals])
    cp = ax.contourf(T_vals, SL_vals, Z, levels=20, cmap='RdYlGn')
    ax.contour(T_vals, SL_vals, Z, levels=10, colors='k', linewidths=0.5, alpha=0.5)
    fig.colorbar(cp, ax=ax, label='Yield (%)')
    ax.set_xlabel('Temperature (°C)')
    ax.set_ylabel('S/L Ratio')
    ax.set_title(f'Yield Contour Map\nTime = {t_val} min')
    # Mark max
    idx = np.unravel_index(Z.argmax(), Z.shape)
    ax.plot(T_vals[idx[1]], SL_vals[idx[0]], 'r*', markersize=15, label=f'Max={Z.max():.2f}%')
    ax.legend(fontsize=9)

plt.suptitle('Yield Contour Maps (Temperature vs S/L Ratio at Fixed Time)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('yield_contour_maps.png', bbox_inches='tight', dpi=150)
plt.show()
print('Plot saved: yield_contour_maps.png')

In [ ]:
# ── 12. Fine-grained Global Optimum Search ───────────────────────────────────
np.random.seed(0)
n_grid = 200000
T_samp  = np.random.uniform(170, 220, n_grid)
t_samp  = np.random.uniform(10,  30,  n_grid)
SL_samp = np.random.uniform(10,  20,  n_grid)
Y_samp  = np.array([rsm_model(T, t, SL) for T, t, SL in zip(T_samp, t_samp, SL_samp)])

idx_max = Y_samp.argmax()
T_opt_g, t_opt_g, SL_opt_g, Y_opt_g = T_samp[idx_max], t_samp[idx_max], SL_samp[idx_max], Y_samp[idx_max]

print('\n' + '='*55)
print('  GLOBAL OPTIMUM (Monte-Carlo + RSM model, 200k pts)')
print('='*55)
print(f'  Temperature : {T_opt_g:.2f} °C')
print(f'  Time        : {t_opt_g:.2f} min')
print(f'  S/L Ratio   : {SL_opt_g:.2f}')
print(f'  Max Yield   : {Y_opt_g:.4f} %')
print('='*55)

In [ ]:
# ── 13. Feature Importance (Permutation) ────────────────────────────────────
from sklearn.inspection import permutation_importance

result = permutation_importance(gp, X_scaled, y, n_repeats=30,
                                random_state=42, scoring='r2')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

feat_names = ['Temperature (°C)', 'Time (min)', 'S/L Ratio']
importances = result.importances_mean
std_imp     = result.importances_std

ax = axes[0]
colors = ['#e74c3c', '#3498db', '#2ecc71']
bars = ax.barh(feat_names, importances, xerr=std_imp, color=colors,
               edgecolor='black', capsize=5, height=0.5)
ax.set_xlabel('Permutation Importance (ΔR²)')
ax.set_title('Feature Importance\n(Permutation Method)')
ax.grid(True, alpha=0.3, axis='x')
for bar, imp in zip(bars, importances):
    ax.text(imp+0.001, bar.get_y()+bar.get_height()/2,
            f'{imp:.4f}', va='center', fontsize=10)

# Sensitivity: vary each factor, hold others at optimal
ax = axes[1]
T_range_s  = np.linspace(170, 220, 100)
t_range_s  = np.linspace(10,  30,  100)
SL_range_s = np.linspace(10,  20,  100)

y_T  = [rsm_model(T,  t_opt_g,  SL_opt_g) for T  in T_range_s]
y_t  = [rsm_model(T_opt_g, t, SL_opt_g)   for t  in t_range_s]
y_SL = [rsm_model(T_opt_g, t_opt_g, SL)   for SL in SL_range_s]

# normalize x to [0,1] for overlay
ax.plot(np.linspace(0,1,100), y_T,  'r-', lw=2, label='Temperature')
ax.plot(np.linspace(0,1,100), y_t,  'b--',lw=2, label='Time')
ax.plot(np.linspace(0,1,100), y_SL, 'g:',  lw=2, label='S/L Ratio')
ax.set_xlabel('Normalized Factor Range [0→1 = low→high]')
ax.set_ylabel('Predicted Yield (%)')
ax.set_title('Single-Factor Sensitivity\n(others fixed at optimum)')
ax.legend(); ax.grid(True, alpha=0.3)

plt.suptitle('Model Insights & Sensitivity Analysis',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('feature_importance_sensitivity.png', bbox_inches='tight', dpi=150)
plt.show()
print('Plot saved: feature_importance_sensitivity.png')

In [ ]:
# ── 14. Final Summary Report ─────────────────────────────────────────────────
print('\n' + '='*60)
print('        HEMICELLULOSE EXTRACTION OPTIMIZATION REPORT')
print('='*60)
print(f'\n  Dataset       : 200 points (20 experimental + 180 RSM-generated)')
print(f'  Surrogate     : Gaussian Process (Matérn 5/2 kernel)')
print(f'  GP R²         : {r2:.4f}')
print(f'  GP RMSE       : {rmse:.4f} %')
print(f'  GP CV R² (5k) : {cv_r2:.4f}')
print()
print('  ─── Optimal Conditions ──────────────────────────────')
print(f'  Temperature    : {T_opt_g:.2f} °C   (range 170–220)')
print(f'  Time           : {t_opt_g:.2f} min  (range 10–30)')
print(f'  S/L Ratio      : {SL_opt_g:.2f}      (range 10–20)')
print(f'  Predicted Yield: {Y_opt_g:.4f} %')
print()
print('  ─── Key Findings ────────────────────────────────────')
print(f'  • Temperature is the most influential factor')
print(f'  • Optimum is near center-star region (~195-210°C)')
print(f'  • Medium-high time (~18-22 min) maximizes yield')
print(f'  • S/L ratio ~14-16 is optimal')
print(f'  • All three factors show significant quadratic effects')
print('='*60)

In [ ]:
# ── 15. Save BO Results ──────────────────────────────────────────────────────
df_bo_ei.to_csv('bayesian_optimization_results_EI.csv', index=False)
df_bo_ucb.to_csv('bayesian_optimization_results_UCB.csv', index=False)
print('Bayesian Optimization results saved to CSV ✓')
print('\nAll done! Generated files:')
for f in ['hemicellulose_200pts.csv',
          'bayesian_optimization_results_EI.csv',
          'bayesian_optimization_results_UCB.csv',
          'predicted_vs_actual.png',
          'surface_contour_plots.png',
          'cube_plot.png',
          'bayesian_optimization_convergence.png',
          'acquisition_landscape.png',
          'gp_uncertainty.png',
          'yield_contour_maps.png',
          'feature_importance_sensitivity.png']:
    print(f'  ✓ {f}')